> **Status: PREPARED — NOT YET EXECUTED**  
> Skeleton notebook ready. Requires dataset from A3_01 and `bgr_model.pt` in `shared/assets/`.  
> **Do NOT execute until A3_01 has been run and the dataset verified.**

# MaCAD S.3 — Assignment 3
## A3-02: No-Grad Prediction — BGR Graph Classification

**Assignment objective:** Apply the pretrained `bgr_model.pt` to the BGR graph dataset
produced in A3-01. Run inference under `torch.no_grad()` using the PyG pipeline from
S06-13B. Visualise predicted building type labels on the graph.

**Course workflow:** S06-13B — GML Predict BGR Graph  
**Primary reference (read-only):** `class_notebooks/S06_graph_ml/S06-13B GML Predict BGR Graph.ipynb`

---

### Required Inputs

| File | Location | Source |
|------|----------|--------|
| `nodes.csv` | `01_input_graph/dataset/` | A3_01 |
| `edges.csv` | `01_input_graph/dataset/` | A3_01 |
| `graphs.csv` | `01_input_graph/dataset/` | A3_01 |
| `meta.yaml` | `01_input_graph/dataset/` | A3_01 |
| `bgr_model.pt` | `shared/assets/course_supporting_files/` | Course provided |

### Outputs

| File | Location | Description |
|------|----------|-------------|
| `predictions.csv` | `03_predictions/` | Predicted BGR label + class probabilities per graph |
| `01_bgr_prediction.png` | `04_visuals/` | Graph coloured by predicted cell type |
| `02_bgr_prediction_detail.png` | `04_visuals/` | Label overlay with colour legend |

### Assignment 3 — Part 2 Checklist

| # | Step | Course API | Status |
|---|------|-----------|--------|
| 1 | Confirm dataset and model present | — | \[ \] Run to confirm |
| 2 | Load dataset via `PyG.ByCSVPath` | S06-13B | \[ \] Run to confirm |
| 3 | Load model via `pyg.LoadModel` | S06-13B | \[ \] Run to confirm |
| 4 | Set split to all-test: `SetHyperparameters(split=(0,0,1))` | S06-13B | \[ \] Run to confirm |
| 5 | Run inference: `pyg.Predict()` | S06-13B | \[ \] Run to confirm |
| 6 | Map predicted labels using BGR label mapping | S06-13B | \[ \] Run to confirm |
| 7 | Export `predictions.csv` | — | \[ \] Run to confirm |
| 8 | Visualise prediction on graph | Topology.Show | \[ \] Run to confirm |
| 9 | Interpret predicted BGR type in spatial terms | — | \[ \] Manual |

---
## BGR Label Mapping (from S06-13B)

| Integer label | BGR type | Description |
|---------------|----------|-------------|
| 0 | Separation | Pilotis fully detached from massing |
| 1 | Separation with Plinth | Pilotis + base podium, no contact |
| 2 | Adherence | Massing touches column grid |
| 3 | Adherence with Plinth | Massing + podium touching columns |
| 4 | Interlock | Massing and columns interdigitate |

---
## 1. Setup

In [ ]:
# !pip install topologicpy torch torch_geometric

from topologicpy.PyG import PyG
from topologicpy.Topology import Topology
from topologicpy.Graph import Graph
from topologicpy.Dictionary import Dictionary
from topologicpy.Vertex import Vertex
from topologicpy.Helper import Helper

import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")

print("topologicpy version:", Helper.Version())

try:
    import torch
    print(f"torch version     : {torch.__version__}")
    print(f"CUDA available    : {torch.cuda.is_available()}")
except ImportError:
    print("WARNING: torch not installed — pip install torch torch_geometric")

In [ ]:
# ── Path configuration ─────────────────────────────────────────────────────
PROJECT_ROOT = "D:/GitHub/GML_Edu"

A3_ROOT      = os.path.join(PROJECT_ROOT, "assignments", "assignment_03_no_grad_evaluation")
DATASET_DIR  = os.path.join(A3_ROOT, "01_input_graph", "dataset")
PREDICTIONS  = os.path.join(A3_ROOT, "03_predictions")
VISUALS_DIR  = os.path.join(A3_ROOT, "04_visuals")

MODEL_PATH   = os.path.join(PROJECT_ROOT, "shared", "assets",
                            "course_supporting_files", "bgr_model.pt")

os.makedirs(PREDICTIONS, exist_ok=True)
os.makedirs(VISUALS_DIR, exist_ok=True)

print("Path check:")
all_ok = True
for label, path in [
    ("nodes.csv    ", os.path.join(DATASET_DIR, "nodes.csv")),
    ("edges.csv    ", os.path.join(DATASET_DIR, "edges.csv")),
    ("graphs.csv   ", os.path.join(DATASET_DIR, "graphs.csv")),
    ("meta.yaml    ", os.path.join(DATASET_DIR, "meta.yaml")),
    ("bgr_model.pt ", MODEL_PATH),
]:
    status = "OK" if os.path.exists(path) else "MISSING"
    if status == "MISSING":
        all_ok = False
    print(f"  {label}: [{status}]")

if not all_ok:
    missing_model = not os.path.exists(MODEL_PATH)
    missing_data  = not os.path.exists(os.path.join(DATASET_DIR, "nodes.csv"))
    if missing_data:
        raise FileNotFoundError("Dataset missing — run A3_01_BGR_Graph_Creation.ipynb first.")
    if missing_model:
        raise FileNotFoundError(
            f"bgr_model.pt not found at:\n  {MODEL_PATH}\n"
            "Obtain from course materials and place in shared/assets/course_supporting_files/"
        )

---
## 2. Validate Dataset and Load

**Course API — S06-13B:**  
`PyG.ByCSVPath(path, level, task, graphLabelType, nodeLabelType, edgeLabelType)`

- `level = "graph"` — graph-level classification task
- `task  = "classification"` — predict discrete BGR type
- `graphLabelType = "categorical"` — matches S06-13B exactly
- `nodeLabelType  = "categorical"` — matches S06-13B exactly
- `edgeLabelType  = "categorical"` — matches S06-13B exactly

A validation cell runs first to confirm the dataset is well-formed before loading.

In [ ]:
# Dataset validation — stop with a clear error if any check fails.
import pandas as pd, yaml as _yaml

NODES_PATH  = os.path.join(DATASET_DIR, "nodes.csv")
GRAPHS_PATH = os.path.join(DATASET_DIR, "graphs.csv")
META_PATH   = os.path.join(DATASET_DIR, "meta.yaml")

_errors = []

# 1. Model file
if not os.path.exists(MODEL_PATH):
    _errors.append(f"bgr_model.pt not found: {MODEL_PATH}")
else:
    print(f"[OK] bgr_model.pt ({os.path.getsize(MODEL_PATH)//1024} KB)")

# 2. graphs.csv exists
if not os.path.exists(GRAPHS_PATH):
    _errors.append(f"graphs.csv not found — run A3_01 first")
else:
    _gdf = pd.read_csv(GRAPHS_PATH)
    print(f"[OK] graphs.csv ({len(_gdf)} rows)")
    if 'label' not in _gdf.columns or _gdf['label'].isna().all():
        _errors.append("graphs.csv: label column is empty — set your BGR assessment manually (see A3_01 Step 8)")
    else:
        _lval = _gdf['label'].dropna().iloc[0]
        print(f"[OK] graphs.csv label: {int(_lval)}")

# 3. nodes.csv — check feature columns
EXPECTED_FEATURES = ["feature_00", "feature_01", "feature_02", "feature_03", "feature_04"]
if not os.path.exists(NODES_PATH):
    _errors.append(f"nodes.csv not found — run A3_01 first")
else:
    _ndf = pd.read_csv(NODES_PATH)
    _present = [f for f in EXPECTED_FEATURES if f in _ndf.columns]
    _missing = [f for f in EXPECTED_FEATURES if f not in _ndf.columns]
    print(f"[OK] nodes.csv ({len(_ndf)} rows)")
    print(f"     feature columns found   : {_present}")
    if _missing:
        _errors.append(f"nodes.csv missing feature columns: {_missing}")
        _errors.append("  → A3_01 must export with NODE_FEATURE_KEYS = feature_00..feature_04")
    else:
        _n_feat = len(_present)
        if _n_feat != 5:
            _errors.append(f"Expected exactly 5 feature columns; found {_n_feat}")
        else:
            print(f"[OK] Exactly 5 feature columns present")

if _errors:
    print()
    print("VALIDATION FAILED — fix these issues before continuing:")
    for _e in _errors:
        print(f"  ERROR: {_e}")
    raise RuntimeError(f"{len(_errors)} validation error(s) — see above")
else:
    print()
    print("Validation passed. Proceed to load dataset.")

In [ ]:
print(f"Loading dataset from: {DATASET_DIR}")
pyg = PyG.ByCSVPath(
    path           = DATASET_DIR,
    level          = "graph",
    task           = "classification",
    graphLabelType = "categorical",
    nodeLabelType  = "categorical",
    edgeLabelType  = "categorical",
)

print(f"PyG dataset loaded.")
print(f"Type: {type(pyg)}")

# Dataset size info (from S06-13B)
try:
    dataset = pyg.dataset
    print(f"Number of graphs  : {len(dataset)}")
    print(f"Node features dim : {dataset.num_node_features}")
    print(f"Edge features dim : {dataset.num_edge_features if hasattr(dataset, 'num_edge_features') else 'N/A'}")
    print(f"Num classes       : {dataset.num_classes}")
except Exception as e:
    print(f"Dataset info not accessible: {e}")

---
## 3. Load Pretrained Model

**Course API — S06-13B:**  
`pyg.LoadModel(model_path)` — loads the pretrained PyTorch model weights.

The model was trained on the full BGR training set and expects:
- Node features: **5-dimensional** one-hot vector (`feature_00`–`feature_04`)
- Graph-level task: 5-class classification (BGR types 0–4)

In [ ]:
print(f"Loading model: {MODEL_PATH}")
pyg.LoadModel(MODEL_PATH)
print("Model loaded.")

try:
    model_obj = pyg.model
    print(f"Model type: {type(model_obj).__name__}")
    n_params = sum(p.numel() for p in model_obj.parameters())
    print(f"Parameters: {n_params:,}")
except Exception:
    print("(model details not directly accessible — standard PyG wrapper)")

---
## 4. Configure for Inference

**Course pattern — S06-13B:**  
`pyg.SetHyperparameters(split=(0.0, 0.0, 1.0), shuffle=False)` — assign all
graphs to the test set. This is the "no-grad" inference mode where we treat
our building as an unseen sample for the pretrained classifier.

In [ ]:
# All graphs go to the test split — no training, no validation
pyg.SetHyperparameters(
    split   = (0.0, 0.0, 1.0),
    shuffle = False,
)
print("Hyperparameters set: split=(0, 0, 1), shuffle=False")
print("All graphs assigned to test set — inference only.")

---
## 5. Run Inference

**Course API — S06-13B:**  
`pyg.Predict()` — runs the model under `torch.no_grad()` on the test split.
Returns a dictionary with keys:
- `'pred'`   : list of predicted integer labels
- `'y_true'` : list of ground-truth labels (may be None/−1 for unlabelled data)
- `'prob'`   : list of class probability arrays (one per graph)

In [ ]:
BGR_LABELS = {
    0: "Separation",
    1: "Separation with Plinth",
    2: "Adherence",
    3: "Adherence with Plinth",
    4: "Interlock",
}

print("Running inference (torch.no_grad)...")
results = pyg.Predict()

pred   = results.get("pred",   [])
y_true = results.get("y_true", [])
probs  = results.get("prob",   [])

print(f"Predictions   : {pred}")
print(f"True labels   : {y_true}")
print()

for i, (p, pr) in enumerate(zip(pred, probs)):
    label = BGR_LABELS.get(int(p), f"Unknown ({p})")
    print(f"Graph {i}: predicted BGR type = {p} → '{label}'")
    if pr is not None:
        prob_str = "  ".join(f"{BGR_LABELS.get(j,'?')[:12]}: {float(pr[j]):.3f}"
                             for j in range(min(5, len(pr))))
        print(f"  Probabilities: {prob_str}")

---
## 6. Export Predictions

In [ ]:
pred_rows = []
for i, (p, pr) in enumerate(zip(pred, probs)):
    row = {"graph_idx": i, "pred_label": int(p),
           "pred_name": BGR_LABELS.get(int(p), "Unknown")}
    if pr is not None:
        for j in range(min(5, len(pr))):
            row[f"prob_{j}_{BGR_LABELS.get(j,'?')[:8].replace(' ','_')}"] = float(pr[j])
    if y_true and i < len(y_true) and y_true[i] is not None:
        row["true_label"] = int(y_true[i])
        row["true_name"]  = BGR_LABELS.get(int(y_true[i]), "Unknown")
    pred_rows.append(row)

pred_df   = pd.DataFrame(pred_rows)
pred_path = os.path.join(PREDICTIONS, "predictions.csv")
pred_df.to_csv(pred_path, index=False)

print(f"predictions.csv : {len(pred_df)} rows  →  {pred_path}")
print()
print(pred_df.to_string(index=False))

---
## 7. Visualise Prediction

In [ ]:
# Class probability bar chart
if probs and len(probs) > 0 and probs[0] is not None:
    prob_arr   = np.array([float(x) for x in probs[0]])
    class_names = [BGR_LABELS.get(j, str(j)) for j in range(len(prob_arr))]
    pred_idx   = int(pred[0])

    fig, ax = plt.subplots(figsize=(9, 4))
    bar_cols = ["tomato" if j == pred_idx else "steelblue" for j in range(len(prob_arr))]
    bars = ax.bar(class_names, prob_arr, color=bar_cols, edgecolor="white", linewidth=0.5)
    ax.set_ylabel("Probability", fontsize=11)
    ax.set_title(
        f"BGR Prediction — '{BGR_LABELS.get(pred_idx, '?')}' (class {pred_idx})",
        fontsize=12
    )
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=20, labelsize=9)
    for bar, val in zip(bars, prob_arr):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    out = os.path.join(VISUALS_DIR, "01_bgr_prediction.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")
else:
    print("No probability data available — predictions only.")

In [ ]:
# Annotated BGR type diagram
if pred:
    pred_label = BGR_LABELS.get(int(pred[0]), f"Label {pred[0]}")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 5)
    ax.axis("off")

    ax.text(5, 4.2, "BGR CLASSIFICATION RESULT",
            ha="center", va="center", fontsize=14, fontweight="bold", color="#222")
    ax.text(5, 3.2, f"Building: Pilotis model (Assignment 3)",
            ha="center", va="center", fontsize=10, color="#555")
    ax.text(5, 2.5, f"Predicted type:",
            ha="center", va="center", fontsize=10, color="#555")
    ax.text(5, 1.6, pred_label,
            ha="center", va="center", fontsize=18, fontweight="bold",
            color="tomato",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#FFF0EE", edgecolor="tomato", lw=1.5))
    ax.text(5, 0.7, f"Class index: {int(pred[0])}    Model: bgr_model.pt    Workflow: S06-13B",
            ha="center", va="center", fontsize=8, color="#888")

    plt.tight_layout()
    out = os.path.join(VISUALS_DIR, "02_bgr_prediction_detail.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")

---
## 8. Spatial Interpretation

Complete the cells below with your architectural reading of the prediction.

**Questions to answer in your submission:**

1. What BGR type was predicted? What does this mean spatially?
2. Does the prediction match your design intent? If not, why might the model disagree?
3. Which class probability was highest? Were any other types close?
4. How would you modify the Rhino model to shift the prediction toward a different BGR type?
5. What are the limitations of graph-level classification for this task?

In [ ]:
# Interpretation scaffold — complete this with your analysis

if pred:
    pred_int   = int(pred[0])
    pred_name  = BGR_LABELS.get(pred_int, "Unknown")
    
    print("=" * 56)
    print("SPATIAL INTERPRETATION")
    print("=" * 56)
    print(f"Predicted BGR type : {pred_int} — '{pred_name}'")
    print()
    print("Spatial meaning of this type:")
    meanings = {
        0: ("Separation",
            "The program massing sits completely separately from the column grid.\n"
            "  Pilotis are free-standing; massing floats above or beside them.\n"
            "  Classic example: Le Corbusier's Unité d'Habitation (pilotis as\n"
            "  independent structural legs, no contact with the massing above)."),
        1: ("Separation with Plinth",
            "As Separation, but a base podium (plinth) is present.\n"
            "  The massing and columns remain separated; only the plinth connects\n"
            "  to the ground plane."),
        2: ("Adherence",
            "The massing directly touches or wraps around some column volumes.\n"
            "  Structural grid is partially embedded in the program envelope."),
        3: ("Adherence with Plinth",
            "As Adherence, but with a ground-level podium formalising the\n"
            "  base connection between columns and massing."),
        4: ("Interlock",
            "Column grid and massing interpenetrate — columns pass through\n"
            "  programmatic volumes, or program wraps tightly around them."),
    }
    if pred_int in meanings:
        print()
        print(meanings[pred_int][1])
    print()
    print("[YOUR ANALYSIS: does this match your design intent? Fill in here.]")
    print("=" * 56)

---
## 9. Verification and Final Checklist

In [ ]:
expected_outputs = [
    os.path.join(PREDICTIONS, "predictions.csv"),
    os.path.join(VISUALS_DIR, "01_bgr_prediction.png"),
    os.path.join(VISUALS_DIR, "02_bgr_prediction_detail.png"),
]

print("Output file check:")
all_ok = True
for path in expected_outputs:
    status = "OK" if os.path.exists(path) else "MISSING"
    if status == "MISSING":
        all_ok = False
    print(f"  {os.path.basename(path):40s}: {status}")

print()
print("Assignment 3 — Part 2 checklist:")
print("  [ ] All required files present (dataset + bgr_model.pt)")
print("  [ ] PyG dataset loaded with correct graph count")
print("  [ ] Model loaded (parameter count printed)")
print("  [ ] Prediction printed with BGR label name")
print("  [ ] Class probabilities printed and make sense")
print("  [ ] predictions.csv exported to 03_predictions/")
print("  [ ] Probability bar chart saved")
print("  [ ] Prediction result card saved")
print("  [ ] Spatial interpretation written (Section 8)")

print()
if all_ok:
    print("Assignment 3 complete.")
else:
    print("Some outputs missing — scroll up to investigate.")

---
## Final Submission Checklist — Assignment 3

**Part 1 (A3_01):**
- [ ] Four Rhino OBJ files exported and placed in `01_input_graph/`
- [ ] CellComplex visualisation shows correctly tagged layers
- [ ] BGR dataset (nodes.csv, edges.csv, graphs.csv, meta.yaml) exported

**Part 2 (A3_02):**
- [ ] Predicted BGR type reported with class index and name
- [ ] Class probabilities bar chart saved
- [ ] `predictions.csv` saved to `03_predictions/`
- [ ] Written interpretation completed in `05_submission_text/`

**Submission note:**  
Keep this assignment completely separate from Assignment 1 (S02/S03 workflow) and
Assignment 2. The BGR graph, the OBJ geometry, and the model are all distinct from
the resident_gen building used in the earlier assignments.

**Reference:**  
Earlier BGR attempt (archived): `archive/superseded_work/A1_01_DoubleL_BGR_Graph_Generation.ipynb`